In [ ]:
import pandas as pd
import numpy as np
import time
import matplotlib.pyplot as plt

In [9]:
# -------------------------
# Generate Sales Dataset
# -------------------------
def generate_sales_data(n_rows):

    df = pd.DataFrame({
        'date': pd.to_datetime(np.random.choice(pd.date_range('2024-01-01', '2024-12-31'), n_rows)),
        'customer_id': np.random.randint(1000, 9999, n_rows),
        'product_category': np.random.choice(['skincare', 'cloths', 'food', 'electronics', 'Sports'], n_rows),
        'amount': np.random.uniform(5, 500, n_rows),
        'region': np.random.choice(['khanyounis', 'gaza', 'rafah', 'deeralbalah'], n_rows)
    })

    # -------------------------
    # Data Quality Issues
    # -------------------------

    # 5% missing values in amount
    missing_count = int(0.05 * n_rows)
    missing_idx = np.random.choice(df.index, missing_count, replace=False)
    df.loc[missing_idx, 'amount'] = np.nan

    # 2% duplicates (ADD method))( row  خارج المليون  duplicate ضفت قيم ال)
    dup_count = int(0.02 * n_rows)
    dup_idx = np.random.choice(df.index, dup_count, replace=False)
    df = pd.concat([df, df.loc[dup_idx]], ignore_index=True)

    return df


# -------------------------
# Generate Dataset
# -------------------------
start = time.time()
sales_df = generate_sales_data(1_000_000)
end = time.time()

print("=== dataset information ===")
print(f"Rows: {len(sales_df):,}")
print(f"Memory: {sales_df.memory_usage().sum() / 1024**2:.2f} MB")
print(f"Generation Time: {end - start:.2f} sec")

# -------------------------
# 1. Total Sales by Region
# -------------------------
print("\n" + "="*50)

region_sales = sales_df.groupby('region')['amount'].sum().reset_index()

print("\n1. Total Sales by Region")
print(region_sales.to_string(index=False, float_format='{:,.2f}'.format))

print("\n" + "="*50)

# -------------------------
# 2. Data Quality Report
# -------------------------
print("\n2. Data Quality")
print(f"Duplicates: {sales_df.duplicated().sum()}")

missing_per_column = sales_df.isnull().sum()

total_missing = missing_per_column.sum()

print(f"Total Missing Values: {total_missing}\n")
print("Missing Values Per Column:\n")

for col in missing_per_column.index:
    print(f"{col}: {missing_per_column[col]}")

# -------------------------
#  3: Memory Usage
# -------------------------
print("\n" + "="*50)
print("3: Memory Usage")

total_memory = sales_df.memory_usage().sum() / 1024**2
print(f"Total Memory: {total_memory:.2f} MB")

print("\nPer Column:\n")

memory_per_column = sales_df.memory_usage() / 1024**2

for col in memory_per_column.index:
    print(f"{col}: {memory_per_column[col]:.2f} MB")

print("\n" + "="*50)


# -------------------------
#  4: Processing Time
# -------------------------
print(" 4: Processing Time")

start = time.time()

sales_df.groupby('product_category')['amount'].mean()

end = time.time()

processing_time = end - start
rows = len(sales_df)
rate = rows / processing_time

print(f"Aggregation time: {processing_time:.4f} seconds\n")
print(f"Rows processed: {rows:,}\n")
print(f"Processing rate: {rate:,.0f} rows/second")




=== dataset information ===
Rows: 1,020,000
Memory: 38.91 MB
Generation Time: 1.40 sec


1. Total Sales by Region
     region        amount
deeralbalah 61,254,430.52
       gaza 61,071,799.41
 khanyounis 61,173,495.06
      rafah 61,254,289.60


2. Data Quality
Duplicates: 20021
Total Missing Values: 50958

Missing Values Per Column:

date: 0
customer_id: 0
product_category: 0
amount: 50958
region: 0

3: Memory Usage
Total Memory: 38.91 MB

Per Column:

Index: 0.00 MB
date: 7.78 MB
customer_id: 7.78 MB
product_category: 7.78 MB
amount: 7.78 MB
region: 7.78 MB

 4: Processing Time
Aggregation time: 0.2089 seconds

Rows processed: 1,020,000

Processing rate: 4,882,831 rows/second


In [8]:
def recommend_tool(rows, columns, update_frequency):

    # small data
    if rows < 100000:
        return "Excel/CSV - Small dataset"

    # large data (batch)
    elif rows >= 1000000 and update_frequency == 'batch':
        return "Spark - Large dataset"

    # real-time data
    elif update_frequency == 'real-time':
        return "Spark Streaming - Real-time processing"

    # medium data
    else:
        return "Pandas - Medium dataset"


# test cases
print(recommend_tool(5000, 10, 'batch'))
print(recommend_tool(5000000, 50, 'batch'))
print(recommend_tool(1000000, 20, 'real-time'))

Excel/CSV - Small dataset
Spark - Large dataset
Spark Streaming - Real-time processing
